# S21 - Confidence scoring for BOM review

**What this is.** A calibrated estimate, per live row, of the probability the engineer will
**accept** the engine's Min/ROP/Max. Target is the review scorecard rule: within 10% on **Max
and ROP**, no absolute floor, Min ignored - imported from `engine_statistical`, never restated.

**What this is not.** It does not gate, sort, rank or auto-approve anything. It is a number
shown next to the row. Measured coverage at >=85% precision is **0%**: there is no threshold on
this data at which auto-passing is safe, and this notebook does not pretend otherwise.

| | |
|---|---|
| Population | live rows only (`route` in active/dying), n=711 across 8 cycles |
| Model | 9-feature L2 logistic regression, scipy L-BFGS-B, Platt-calibrated |
| Success metric | **calibration** (ECE, reliability), not AUC |
| Expected walk-forward AUC | ~0.70-0.72. **Materially above 0.75 means leakage - stop.** |
| Dependencies | numpy, pandas, scipy, matplotlib. No sklearn, no LightGBM. |

Features, the target and every numeric live in `analysis/s21_confidence_features.py`, so a
future backend scoring path cannot drift from what was trained here. This notebook is the
evaluation narrative only.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "backend").is_dir())
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "backend" / "app")]

import engine_statistical  # the production sizing engine
import s21_confidence_features as F
from common import OUT

pd.set_option("display.width", 160)
print("engine", engine_statistical.MODEL_VERSION,
      "| agree_tol", engine_statistical.AGREE_TOL,
      "| graded on", engine_statistical.AGREE_FIELDS)
print("features", len(F.FEATURES), "->", ", ".join(F.FEATURES))

## 1. Data and target

`s20_payloads.pkl` is one row per reviewed item per month, carrying the engine's own inputs
plus the engineer's final decision (`_final_max` / `_final_rop`). `F.build` re-runs the shipped
engine over it and grades every row with the scorecard rule.

Refresh it with `python analysis/s20_diverge_features.py` (needs DB access) if the month list
below is stale.

In [ ]:
payloads = F.load_payloads()
X, y, live, month = F.build(payloads)
months = sorted(set(month))

print(f"{len(X):,} rows across {len(months)} cycles: {months[0]} .. {months[-1]}")
print(f"live rows (active + dying)   {int(live.sum()):,}")
print(f"base acceptance rate (live)  {y[live].mean() * 100:.1f}%")
print(f"has_prior coverage (live)    {X.has_prior.values[live].mean() * 100:.1f}%")
print()

per_month = pd.DataFrame({"m": month, "y": y, "hp": X.has_prior.values})[live].groupby("m").agg(
    n=("y", "size"), base_rate=("y", "mean"), has_prior=("hp", "mean"))
per_month.assign(base_rate=(per_month.base_rate * 100).round(1),
                 has_prior=(per_month.has_prior * 100).round(1))

The base rate is not stable: it runs from **15.7%** (2025-03) to **44.0%** (2026-01). That one
fact is why the model is recalibrated every cycle rather than fitted once - section 3 shows
what ignoring it costs.

`has_prior` climbs as item history fills in. The early cycles are mostly cold-start, which is
exactly the regime the missing-indicator encoding exists for.

## 2. Walk-forward evaluation

Train on every earlier cycle, predict the current one. **Never random K-fold** - random folds
put future months into the training set, which read 0.741 instead of the honest 0.717 on this
data. Standardisation uses the **training fold's** mean and std only, for the same reason.

In [ ]:
A = X.values
folds = []
for i, mo in enumerate(months):
    if i == 0:
        continue                                   # no earlier data to train on
    te = live & (month == mo)
    tr = live & np.isin(month, months[:i])
    if te.sum() < 5 or y[tr].sum() < 5 or (1 - y[tr]).sum() < 5:
        print(f"{mo}  skipped - fold has too few of a class")
        continue
    mu, sd = F.standardise(A[tr])                  # training fold only
    w = F.fit_logistic((A[tr] - mu) / sd, y[tr])
    Z = (A[te] - mu) / sd
    folds.append(pd.DataFrame({"month": mo, "y": y[te],
                               "z": F.log_odds(w, Z), "p": F.predict(w, Z)}))
    print(f"{mo}  test n={int(te.sum()):4d}  train n={int(tr.sum()):4d}  "
          f"AUC {F.auc(y[te], folds[-1].z.values):.3f}")

oof = pd.concat(folds, ignore_index=True)
y_oof, z_oof = oof.y.values, oof.z.values
POOLED_AUC = F.auc(y_oof, z_oof)
print(f"\npooled walk-forward AUC {POOLED_AUC:.4f}   "
      f"Brier {F.brier(y_oof, oof.p.values):.4f}   n={len(oof)}")

# The stop signal. Above the band the likely cause is leakage in the
# prior-cycle feature, not a better model.
assert 0.68 <= POOLED_AUC <= 0.75, f"AUC {POOLED_AUC:.3f} outside the measured band"

Discrimination is modest and expected to be: **~0.72**. Acceptance is driven largely by
judgement the engine cannot read (the S20 justification analysis), so ranking power is capped
here. The deliverable is the probability being *honest*, which is section 3.

## 3. Calibration

Platt scaling **is** a logistic regression on the raw log-odds, so it reuses the same fitter.
Fitted on out-of-fold predictions only - in-sample calibration produces a perfect diagram that
means nothing. Sigmoid, not isotonic: isotonic needs ~1000+ samples and at this size it
memorises.

In [ ]:
platt = F.fit_logistic(z_oof.reshape(-1, 1), y_oof)      # -> [b, a]
p_raw = oof.p.values
p_cal = F.predict(platt, z_oof.reshape(-1, 1))

print(f"platt   a {platt[1]:+.4f}   b {platt[0]:+.4f}")
print(f"Brier   raw {F.brier(y_oof, p_raw):.4f}  ->  calibrated {F.brier(y_oof, p_cal):.4f}")
print(f"ECE     raw {F.ece(y_oof, p_raw):.4f}  ->  calibrated {F.ece(y_oof, p_cal):.4f}")
assert F.ece(y_oof, p_cal) < 0.10, "calibration did not land"

rel = F.reliability(y_oof, p_cal)
rel.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot([0, 1], [0, 1], "--", lw=1, color="grey", label="perfectly calibrated")
ax.plot(rel.predicted, rel.observed, "o-", color="#1f77b4", label="model (out-of-fold)")
for _, r in rel.dropna().iterrows():
    ax.annotate(f"n={int(r.n)}", (r.predicted, r.observed),
                textcoords="offset points", xytext=(6, -10), fontsize=8, color="grey")
ax.set(xlabel="predicted acceptance probability", ylabel="observed acceptance rate",
       xlim=(0, 1), ylim=(0, 1), title="S21 reliability (5 bins, out-of-fold)")
ax.legend(loc="upper left", fontsize=8)
fig.savefig(OUT / "s21_reliability.png", dpi=150, bbox_inches="tight")
print("wrote", OUT / "s21_reliability.png")
plt.show()

The top two bins hold a handful of rows each; their observed rate is noise, not a finding. The
mass of the data sits below 0.6, which is the honest shape of a 31% base rate.

In [ ]:
bymo = oof.assign(p=p_cal).groupby("month").agg(
    n=("y", "size"), predicted=("p", "mean"), actual=("y", "mean"))
bymo["gap_pp"] = ((bymo.predicted - bymo.actual) * 100).round(1)
bymo.to_csv(OUT / "s21_calibration.csv")
print("wrote", OUT / "s21_calibration.csv")
bymo.round(3)

**This table is the argument for recalibrating every cycle.** Pooled over all months the model
is well calibrated, but no single month is: 2025-03 is over-predicted by roughly 9 points and
2026-01 under-predicted by roughly 16. The cycle-level acceptance rate moves faster than the
features do, so the fit must be refreshed each run - and the reviewer must see the cycle's own
base rate beside the score, or a 30% will read as "low" in a 44% month.

## 4. What the model learned

In [ ]:
mu_all, sd_all = F.standardise(A[live])
w_all = F.fit_logistic((A[live] - mu_all) / sd_all, y[live])

coefs = pd.Series(w_all[1:], index=F.FEATURES).sort_values(key=abs, ascending=False)
print(f"intercept {w_all[0]:+.4f}   (standardised features, so coefficients compare directly)")
print(coefs.round(4).to_string())

# Measured directions. A positive sign on either means the feature or the target
# was built wrong, not that the data changed its mind.
assert coefs["gap_prior_decision"] < 0, "bigger gap vs the prior decision must lower acceptance"
assert coefs["delta_abs_vs_current"] < 0, "bigger change vs current must lower acceptance"

The distance features dominate and both point the same way: the further the engine moves a part
from where it already sits, or from what the engineer last approved, the less likely that move
is accepted. `gap_prior_decision` alone separates 58.3% acceptance at zero gap from 23.3% above
a 50% gap.

## 5. Persist

JSON, not a pickle. The artifact is ~15 numbers: it diffs cleanly in git when a coefficient
moves, it has no library-version coupling, and the backend can score a row with one dot product
and no ML import:

```
z = ((x - feature_means) / feature_stds) @ coef + intercept
p = sigmoid(platt_a * z + platt_b)
```

`agree_tol` and `engine_model_version` are persisted deliberately: if either changes, this
model was trained against a different target and must be retrained. Without them that mismatch
is silent.

In [ ]:
art = {
    "features": F.FEATURES,
    "coef": w_all[1:].tolist(), "intercept": float(w_all[0]),
    "feature_means": mu_all.tolist(), "feature_stds": sd_all.tolist(),
    "platt_a": float(platt[1]), "platt_b": float(platt[0]),
    "trained_at": pd.Timestamp.utcnow().isoformat(),
    "n_train": int(live.sum()), "base_rate": float(y[live].mean()),
    "walk_forward_auc": float(POOLED_AUC),
    "ece_calibrated": float(F.ece(y_oof, p_cal)),
    "engine_model_version": engine_statistical.MODEL_VERSION,
    "agree_tol": engine_statistical.AGREE_TOL,
    "agree_fields": list(engine_statistical.AGREE_FIELDS),
}
ARTIFACT = OUT / "s21_confidence_model.json"
ARTIFACT.write_text(json.dumps(art, indent=2), encoding="utf-8")
print("wrote", ARTIFACT)
print(json.dumps({k: v for k, v in art.items() if k != "features"}, indent=2)[:900])

In [ ]:
# Reload as a cold consumer would: JSON in, five probabilities out, no F.
m = json.loads(ARTIFACT.read_text(encoding="utf-8"))
assert m["features"] == F.FEATURES, "artifact feature order drifted from the module"

sample = X.values[live][:5]
z = ((sample - np.array(m["feature_means"])) / np.array(m["feature_stds"])
     ) @ np.array(m["coef"]) + m["intercept"]
p_reloaded = 1 / (1 + np.exp(-(m["platt_a"] * z + m["platt_b"])))

p_notebook = F.predict(platt, F.log_odds(w_all, (sample - mu_all) / sd_all).reshape(-1, 1))
print(np.c_[p_reloaded, p_notebook].round(6))
assert np.allclose(p_reloaded, p_notebook, atol=1e-9), "artifact does not reproduce the notebook"
print("\nartifact reproduces notebook scores to 1e-9")